In [ ]:
import os
import yaml
import numpy as np
from tqdm import tqdm

In [ ]:
SAMPLE_DIR = './synthetic_samples/'
TRUTH_DIR = './synthetic_ground_truth/'
DIST_NAMES = ['mixture4']

p_test = [1e-4, 1e-5, 1e-6]
p_all = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13, 1e-14, 1e-15]

### RESTK

In [ ]:
from memik import memik

TIGHTNESS_DIR = './synthetic_tightness_memik/'
os.makedirs(TIGHTNESS_DIR, exist_ok=True)

with tqdm(DIST_NAMES) as pbar:
    for dist_name in pbar:
        pbar.set_description(f"Processing {dist_name}")
        pbar.refresh()
        has_negative = False
        samples = np.load(os.path.join(SAMPLE_DIR, dist_name + '.npy'), allow_pickle=True)
        minimum = np.min(samples)
        if minimum <= 0:
            has_negative = True
            samples = samples - minimum + 1
        ground_truth_dict = np.load(os.path.join(TRUTH_DIR, dist_name + '.npy'), allow_pickle=True).item()

        pbar.set_postfix_str('restk')
        pbar.refresh()
        max_k = memik.restk(samples, 1, 150, 1000, p_test, p_all, 1, int(1e3))
        for k, v in max_k.items():
            if k <= 0:
                max_k[k] = 1
        print(f"max_k: {max_k}")
        pbar.set_postfix_str('memik')
        pbar.refresh()
        memik_dict = memik.memik(samples, p_all, max_k, 1)

        if has_negative:
            for key in memik_dict.keys():
                memik_dict[key] = (memik_dict[key][0] + minimum - 1, memik_dict[key][1])

        tightness_dict = {}
        for p in p_all:
            tightness_dict[p] = (float(memik_dict[p][0] / ground_truth_dict[p]), float(memik_dict[p][0]), float(memik_dict[p][1]))

        with open(os.path.join(TIGHTNESS_DIR, dist_name + '.yaml'), 'w') as fw:
            yaml.dump(tightness_dict, fw, default_flow_style=False)

### ARCTAN

In [ ]:
from atan import atan_memik

TIGHTNESS_DIR = './synthetic_tightness_atan/'
os.makedirs(TIGHTNESS_DIR, exist_ok=True)

with tqdm(DIST_NAMES) as pbar:
    for dist_name in pbar:
        pbar.set_description(f"Processing {dist_name}")
        pbar.refresh()
        has_negative = False
        samples = np.load(os.path.join(SAMPLE_DIR, dist_name + '.npy'), allow_pickle=True)
        minimum = np.min(samples)
        if minimum <= 0:
            has_negative = True
            samples = samples - minimum + 1
        ground_truth_dict = np.load(os.path.join(TRUTH_DIR, dist_name + '.npy'), allow_pickle=True).item()

        mid = (np.min(samples) + np.max(samples)) / 2
        d_list = np.logspace(np.log10(mid / 50), np.log10(np.max(samples) * 5000), 13)

        pbar.set_postfix_str('restk')
        pbar.refresh()
        max_k = atan_memik.restk(samples, 1, 150, 500, p_test, p_all, d_list, 1, int(1e3))
        for d, kdict in max_k.items():
            for k, v in kdict.items():
                if k <= 0:
                    max_k[d][k] = 1
        pbar.set_postfix_str('memik')
        pbar.refresh()
        memik_dict = atan_memik.memik(samples, p_all, max_k, 1, d_list)

        if has_negative:
            for key in memik_dict.keys():
                memik_dict[key] = (memik_dict[key][0] + minimum - 1, memik_dict[key][1], memik_dict[key][2])

        tightness_dict = {}
        for p in p_all:
            tightness_dict[p] = (float(memik_dict[p][0] / ground_truth_dict[p]), float(memik_dict[p][0]), float(memik_dict[p][1]), float(memik_dict[p][2]))

        with open(os.path.join(TIGHTNESS_DIR, dist_name + '.yaml'), 'w') as fw:
            yaml.dump(tightness_dict, fw, default_flow_style=False)

### TANH

In [ ]:
from tanh import tanh_memik

TIGHTNESS_DIR = './synthetic_tightness_tanh/'
os.makedirs(TIGHTNESS_DIR, exist_ok=True)

with tqdm(DIST_NAMES) as pbar:
    for dist_name in pbar:
        pbar.set_description(f"Processing {dist_name}")
        pbar.refresh()
        has_negative = False
        samples = np.load(os.path.join(SAMPLE_DIR, dist_name + '.npy'), allow_pickle=True)
        minimum = np.min(samples)
        if minimum <= 0:
            has_negative = True
            samples = samples - minimum + 1
        ground_truth_dict = np.load(os.path.join(TRUTH_DIR, dist_name + '.npy'), allow_pickle=True).item()

        mid = (np.min(samples) + np.max(samples)) / 2
        d_list = np.logspace(np.log10(mid / 50), np.log10(np.max(samples) * 1000), 13)

        pbar.set_postfix_str('restk')
        pbar.refresh()
        max_k = tanh_memik.restk(samples, 1, 150, 100, p_test, p_all, d_list, 1, int(1e3))
        for d, kdict in max_k.items():
            for k, v in kdict.items():
                if k <= 0:
                    max_k[d][k] = 1
        pbar.set_postfix_str('memik')
        pbar.refresh()
        memik_dict = tanh_memik.memik(samples, p_all, max_k, 1, d_list)

        if has_negative:
            for key in memik_dict.keys():
                memik_dict[key] = (memik_dict[key][0] + minimum - 1, memik_dict[key][1], memik_dict[key][2])

        tightness_dict = {}
        for p in p_all:
            tightness_dict[p] = (float(memik_dict[p][0] / ground_truth_dict[p]), float(memik_dict[p][0]), float(memik_dict[p][1]), float(memik_dict[p][2]))

        with open(os.path.join(TIGHTNESS_DIR, dist_name + '.yaml'), 'w') as fw:
            yaml.dump(tightness_dict, fw, default_flow_style=False)